# Hyperparameter Tuning

Hyperparameters are the parameters in a machine learning model that the user chooses.

For example:
* learning rate
* step size
* tree depth
* type of optimizer
* number of layers
* kernel size

Each model has a unique set of hyperparameters that set the architecture of the model and the training process. Tuning hyperparameters involves training a baseline model to have a reference point for metrics, then manually improving the hyperparameters to achieve better metrics than the baseline model. This training and tuning is all done on the training and validation sets, then oficially assessed on the test set, which has not been used for any training.

Methods to tune hyperparameters include manual search, grid search, and more advanced methods such as Bayesian optimization and successive halving. This workshop will focus on manual search.



In this workshop, students will:
* choose a baseline loss function and optimizer for a fully connected neural network
* find the 2-3 most important hyperparameters of the model as a whole
* fit a baseline model to a simple classification dataset
* tune hyperparameters and train on same dataset
* compare performance metrics from the baseline and final tuned model

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import v2
import torch.nn.functional as F

# Device selection: CUDA → MPS (Apple Silicon) → CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device.type}\n")

if device.type == "cuda":
    num_gpus = torch.cuda.device_count()
    print(f"Found {num_gpus} CUDA device(s):")
    for idx in range(num_gpus):
        print(f"  [{idx}] {torch.cuda.get_device_name(idx)}")
elif device.type == "mps":
    print("Apple Metal (MPS) device available")
else:
    print("Running on CPU")

print("\nPython:", platform.python_version(), "| PyTorch:", torch.__version__)

# Instructions

Use the PyTorch Documentation to research some loss functions and optimizers. Choose one each and find the 2-3 most important hyperparameters of the model as a whole. After finding the hyperparameters, fit a baseline model to the CIFAR10 dataset. Assess the performance metrics of the baseline model. Fit 3 other models, tuning the hyperparameters you found by changing only one value at a time. Assess the performance metrics of each improved model as it trains. Ensure you are only training on the train set and assessing the performance metrics on the val set. Once you have trained all 4 models, perform a final evaluation using the baseline model and the best performing tuned model on the unsed test set.



# Read in Data and Preprocess

The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images.

The classes are: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
trainval_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_size = 40000
val_size   = len(trainval_data) - train_size
train_data, val_data = random_split(trainval_data, [train_size, val_size], generator=torch.Generator().manual_seed(42))

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

batch_size = 64

train_dataloader = DataLoader(train_data, batch_size=batch_size)
val_dataloader = DataLoader(val_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [ ]:
labels_map = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck",
}

figure = plt.figure(figsize=(8, 8))
cols, rows = 5, 5
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(train_data), size=(1,)).item()
    img, label = train_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.permute(1, 2, 0))
plt.show()

Note: the targets have been read in as integers, not all loss functions need integer targets.

In [ ]:
# shape of images
images, labels = next(iter(train_dataloader))
print("Image batch shape: ", images.size())
print("Label batch shape: ", labels.size())


# Number of Layers, Loss Function, and Optimizer

Choose the number of hidden layers, loss function, and optimizer.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, output_size, hidden_sizes: list):
        super().__init__()

        # list of layers for the model
        layers = [nn.Flatten()]

        # loop through number of hidden layers
        current_input_size = input_size
        for h_size in hidden_sizes:
            layers.append(nn.Linear(current_input_size, h_size))
            layers.append(nn.ReLU())
            current_input_size = h_size

        # final output layer
        layers.append(nn.Linear(current_input_size, output_size))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        logits = self.network(x)
        return logits

In [ ]:
# Calculate input and output sizes
input_size = images.shape[1] * images.shape[2] * images.shape[3]
output_size = len(labels_map)

# Load model to device and print architecture
model_baseline = NeuralNetwork(input_size, output_size, hidden_sizes=[HIDDEN_1, HIDDEN_2, ETC]).to(device)
print(model_baseline)

# Count parameters in the entire model
total_params = sum(p.numel() for p in model_baseline.parameters())
print("\nTotal parameters:", total_params)

# Choose loss and optimizer
loss_fn = nn.LOSS()
optimizer = torch.optim.OPTIMIZER()

# Train and Test Functions

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    correct = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def eval_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    correct = 0

    # Freeze gradients
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    correct /= size
    print(f"Accuracy: {(correct):0.4f}\nLoss: {(loss):0.4f}\n")


def train_model(epochs, train_dataloader, val_dataloader, model, loss_fn, optimizer):
  for t in range(epochs):
    print(f"Epoch {t+1}\n")

    # Train model on train data (update params)
    train_loop(train_dataloader, model, loss_fn, optimizer)

    # Check train accuracy for this epoch
    print("\nTrain:")
    eval_loop(train_dataloader, model, loss_fn)

    # Check val accuracy for this epoch
    print("Val:")
    eval_loop(val_dataloader, model, loss_fn)

    print("-------------------------------\n")
  print("Done!")


def eval_model(dataloader, model, loss_fn):
  # Put model in evaluation mode
  model.eval()

  all_y_true = []
  all_y_pred = []

  # Run test data through model again, freezing gradients since only doing a forward pass
  with torch.no_grad():
    for X, y in test_dataloader:
      X, y = X.to(device), y.to(device)
      pred = model(X)
      # Get predicted class indices (argmax of logits)
      y_pred = pred.argmax(dim=1).cpu().numpy()
      y_true = y.cpu().numpy()

      all_y_pred.append(y_pred)
      all_y_true.append(y_true)


  all_y_pred = np.concatenate(all_y_pred)
  all_y_true = np.concatenate(all_y_true)

  print(classification_report(all_y_true, all_y_pred))

Choose number of epochs and train baseline model.

In [ ]:
train_model(EPOCHS, train_dataloader, val_dataloader, model_baseline, loss_fn, optimizer)

# Change a Hyperparameter Then Train Again

Repeat this 2-3 times, training new models each time. Note which combination of hyperparameters yields the highest train/val accuracy. This will be your final model.

In [ ]:
# Load model to device and print architecture
model_next = NeuralNetwork(input_size, output_size, hidden_sizes=[HIDDEN_1, HIDDEN_2, ETC]).to(device)
print(model_next)

# Count parameters in the entire model
total_params = sum(p.numel() for p in model_next.parameters())
print("\nTotal parameters:", total_params)

# Choose loss and optimizer
loss_fn = nn.LOSS()
optimizer = torch.optim.OPTIMIZER()

In [ ]:
train_model(EPOCHS, train_dataloader, val_dataloader, model_next, loss_fn, optimizer)

# Evaluate Baseline Model Performance Against Final Model

In [ ]:
eval_model(test_dataloader, model_baseline, loss_fn_BASE)
eval_model(test_dataloader, model_FINAL, loss_fn_FINAL)